# 09 · Compile QAOA for a processor topology

## Question

What changes when the logical QAOA circuit is compiled for a real quantum processor architecture?

## Concept

**Informatics view.** An algorithm goes through a compiler to run on a machine architecture. Here the logical QAOA circuit specifies the algorithm's gates, without assuming where the qubits sit. The Qiskit transpiler maps it to physical qubits, routes interactions onto allowed links, and translates gates to the target's native gate set, the instructions available on that processor.

**Physics / quantum view.** A topology is the graph of allowed physical interactions. If two quantum states assigned to distant sites must interact, routing can move them using SWAP gates, which exchange the states of two qubits. On a bidirectional CX link, a SWAP can be implemented with three CX gates. The final layout records where each logical state ends up. [Qiskit transpiler stages](https://quantum.cloud.ibm.com/docs/en/qiskit/transpiler-stages)

## Worked example

Use the illustrative 4×2 chain from [Notebook 06](06_coupled_policy_map.ipynb) and the compact binary QUBO from [Notebook 08](08_encoding_comparison.ipynb). The original one-hot model provides a comparison. Both use the same score matrix, chain edges, and switching weight 2. Exact enumeration gives map 1112 with score 13. Here the digits are one-based policy labels.

The target is a **synthetic superconducting-style model** built with Qiskit's [GenericBackendV2](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.providers.fake_provider.GenericBackendV2). It has eight sites in a line, bidirectional nearest-neighbor CX, and one-qubit gates RZ, SX and X. The topology and basis are chosen for this experiment. Calibration and noise are omitted. Euro-Q-Exa-specific compilation is a later step once a reproducible target is available.

## Exercise / prediction

With identity placement, can physical sites 0 and 3 interact directly? Predict whether the compact chain needs a SWAP on this target. Then predict which encoding has more native CX gates after decomposition and routing.

In [1]:
from itertools import product
import platform

import numpy as np
import qiskit
import qiskit_aer
from IPython.display import Markdown, display
from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.quantum_info import Operator, Statevector
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import BasicSwap
from qiskit_aer import AerSimulator

J = np.array([[4, 0], [1, 4], [4, 0], [0, 6]])
n = len(J)
edges = [(i, i + 1) for i in range(n - 1)]
beta = 2
A = int(J.sum()) + 1
scores = {p: int(J[np.arange(n), p].sum()) -
          beta * sum(p[i] != p[k] for i, k in edges)
          for p in product(range(2), repeat=n)}
assert max(scores, key=scores.get) == (0, 0, 0, 1) and max(scores.values()) == 13

# Compact switching cost on each edge is beta*(x_i+x_k-2*x_i*x_k).
compact_linear = (J[:, 0] - J[:, 1]).astype(float)
for i, k in edges:
    compact_linear[[i, k]] += beta
compact_pairs = {edge: -2*beta for edge in edges}
onehot_pairs = {(2*i, 2*i + 1): 2*A for i in range(n)}
onehot_pairs.update({(2*i + j, 2*k + 1 - j): beta for i, k in edges for j in range(2)})
models = {
    "Compact": (-J[:, 0].sum(), compact_linear, compact_pairs),
    "One-hot": (n*A, -J.ravel().astype(float) - A, onehot_pairs),
}

def qaoa(fields, zz, gamma, theta):
    circuit = QuantumCircuit(len(fields))
    circuit.h(range(len(fields)))
    for i, field in enumerate(fields):
        if field:
            circuit.rz(2*gamma*field, i)
    for (i, k), coupling in zz.items():
        circuit.rzz(2*gamma*coupling, i, k)
    circuit.rx(2*theta, range(len(fields)))
    return circuit

### Verify the logical circuits

Convert each QUBO using $x=(1-Z)/2$. Check the energy of every binary string and the score of every valid map. Choose angles separately for each encoding with the same 32×24 ideal angle grid and use the same 8,192-shot ideal Aer run as the earlier one-hot example. Once chosen, freeze the angles for every compilation level.

In [2]:
logical = {}
seed, shots = 42, 8192
aer = AerSimulator()
for name, (constant, linear, pairs) in models.items():
    width = len(linear)
    offset = constant + linear.sum()/2
    fields = -linear/2
    zz = {pair: weight/4 for pair, weight in pairs.items()}
    for (i, k), weight in zz.items():
        offset += weight
        fields[[i, k]] -= weight

    bits = (np.arange(2**width)[:, None] >> np.arange(width)) & 1
    spins = 1 - 2*bits
    binary_energy = constant + bits @ linear
    energy = offset + spins @ fields
    for (i, k), weight in pairs.items():
        binary_energy += weight*bits[:, i]*bits[:, k]
        energy += zz[i, k]*spins[:, i]*spins[:, k]
    np.testing.assert_array_equal(energy, binary_energy)

    rows = bits.reshape(-1, n, width//n)
    feasible = (rows.sum(axis=2) == 1).all(axis=1) if name == "One-hot" else np.ones(len(bits), dtype=bool)
    decoded = rows.argmax(axis=2) if name == "One-hot" else bits
    assert {tuple(p): -e for p, e in zip(decoded[feasible], energy[feasible], strict=True)} == scores
    ground = energy == energy.min()
    assert np.all(feasible[ground]) and energy.min() == -13

    grid = product(np.linspace(0, 2*np.pi, 32, endpoint=False),
                   np.linspace(0, np.pi, 24, endpoint=False))
    _, gamma, theta = min((float(Statevector.from_instruction(qaoa(fields, zz, g, t))
                                .probabilities() @ energy), g, t) for g, t in grid)
    circuit = qaoa(fields, zz, gamma, theta)
    logical[name] = circuit
    measured = circuit.copy()
    measured.measure_all()
    counts = aer.run(measured, shots=shots, seed_simulator=seed).result().get_counts()
    sample = np.array([counts.get(f"{i:0{width}b}", 0) for i in range(len(bits))])
    assert sample.sum() == shots
    print(f"{name}: γ={gamma:.6f}, θ={theta:.6f}; feasible {sample[feasible].sum()/shots:.2%}, "
          f"optimal {sample[ground].sum()/shots:.2%}")

print(f"Python {platform.python_version()}, Qiskit {qiskit.__version__}, Aer {qiskit_aer.__version__}")
print(f"{aer.name}, p=1, {shots} shots, simulator seed {seed}")

Compact: γ=0.196350, θ=2.487094; feasible 100.00%, optimal 19.62%


One-hot: γ=6.086836, θ=2.748894; feasible 50.50%, optimal 5.87%
Python 3.15.2, Qiskit 2.6.2, Aer 0.18.4
aer_simulator, p=1, 8192 shots, simulator seed 42


### Show one routing step

The one-hot QUBO has an RZZ on logical qubits 0 and 3. The identity placement puts them at the endpoints of the path 0—1—2—3. BasicSwap moves the state from site 0 through the intermediate sites so the RZZ can act on adjacent sites. This diagram stops before translating the RZZ and SWAPs into native gates.

In [3]:
line = CouplingMap.from_line(8)
basis = ["rz", "sx", "x", "cx"]
target = GenericBackendV2(8, basis_gates=basis.copy(), coupling_map=line,
                          seed=seed, noise_info=False)
print("Physical connectivity: 0—1—2—3—4—5—6—7 (both directions)")
print("Native gate basis:", ", ".join(basis))

interaction = QuantumCircuit(4)
interaction.rzz(0.5, 0, 3)
routed_demo = PassManager(BasicSwap(CouplingMap.from_line(4))).run(interaction)
assert routed_demo.count_ops().get("swap", 0) == 2
print("Logical:\n", interaction.draw(output="text"))
print("Routed, before native translation:\n", routed_demo.draw(output="text"))

Physical connectivity: 0—1—2—3—4—5—6—7 (both directions)
Native gate basis: rz, sx, x, cx
Logical:
                
q_0: ─■────────
      │        
q_1: ─┼────────
      │        
q_2: ─┼────────
      │ZZ(0.5) 
q_3: ─■────────
               
Routed, before native translation:
                          
q_0 -> 0 ─X──────────────
          │              
q_1 -> 1 ─X──X───────────
             │           
q_2 -> 2 ────X──■────────
                │ZZ(0.5) 
q_3 -> 3 ───────■────────
                         


## Result · compile both encodings

Use identity initial placement, BasicSwap routing and transpiler seed 42 for levels 0, 1 and 3. Retaining one router across levels isolates changes from the other compiler stages. The SWAP count is observed immediately after routing, before native translation. Compare against the same native basis on an all-connected target to estimate the net CX overhead of limited connectivity. Optimizations may cancel gates, so this difference need not equal three times the SWAP count.

Active sites are physical wires touched by gates. The compiled circuit object can also contain idle wires. Layout lists below give physical sites in logical-qubit order. Measurements are excluded from these counts.

In [4]:
def resources(circuit):
    gates = [item for item in circuit.data if item.operation.name != "barrier"]
    active = {circuit.find_bit(q).index for item in gates for q in item.qubits}
    return (circuit.depth(), len(gates),
            sum(len(item.qubits) == 1 for item in gates),
            sum(len(item.qubits) == 2 for item in gates), len(active))

all_connected = GenericBackendV2(8, basis_gates=basis.copy(),
                                 coupling_map=CouplingMap.from_full(8),
                                 seed=seed, noise_info=False)
lines = ["| Encoding | Level | Logical qubits | Active sites | Depth before → after | Total gates before → after | 1Q before → after | 2Q before → after | Routing SWAPs | Extra CX vs all-to-all |",
         "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
layouts = ["| Encoding | Level | Initial placement | Final placement | Native gates used |",
           "|---|---:|---|---|---|"]
compiled_results = {}
for name, circuit in logical.items():
    before = resources(circuit)
    for level in (0, 1, 3):
        swap_counts = []

        def observe_routing(**info):
            if isinstance(info["pass_"], BasicSwap):
                swap_counts.append(info["dag"].count_ops().get("swap", 0))

        options = dict(initial_layout=list(range(circuit.num_qubits)), routing_method="basic",
                       seed_transpiler=seed, optimization_level=level, qubits_initially_zero=False)
        mapped = transpile(circuit, backend=target, callback=observe_routing, **options)
        baseline = transpile(circuit, backend=all_connected, **options)
        routing_swaps = max(swap_counts, default=0)  # Skip means the layout already fits.
        for item in mapped.data:
            physical = tuple(mapped.find_bit(q).index for q in item.qubits)
            assert target.target.instruction_supported(operation_name=item.operation.name, qargs=physical)
        expected = QuantumCircuit(mapped.num_qubits)
        expected.compose(circuit, qubits=range(circuit.num_qubits), inplace=True)
        assert Operator.from_circuit(mapped).equiv(Operator(expected))
        after = resources(mapped)
        overhead = mapped.count_ops().get("cx", 0) - baseline.count_ops().get("cx", 0)
        compiled_results[name, level] = (before, after, routing_swaps, overhead)
        transitions = " | ".join(f"{a} → {b}" for a, b in zip(before[:4], after[:4]))
        lines.append(f"| {name} | {level} | {circuit.num_qubits} | {after[4]} | {transitions} | "
                     f"{routing_swaps} | {overhead:+d} |")
        initial = mapped.layout.initial_index_layout(filter_ancillas=True)
        final = mapped.layout.final_index_layout(filter_ancillas=True)
        native = ", ".join(sorted(mapped.count_ops()))
        layouts.append(f"| {name} | {level} | {initial} | {final} | {native} |")
display(Markdown("\n".join(lines)))
display(Markdown("\n".join(layouts)))

| Encoding | Level | Logical qubits | Active sites | Depth before → after | Total gates before → after | 1Q before → after | 2Q before → after | Routing SWAPs | Extra CX vs all-to-all |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Compact | 0 | 4 | 4 | 6 → 18 | 15 → 45 | 12 → 39 | 3 → 6 | 0 | +0 |
| Compact | 1 | 4 | 4 | 6 → 17 | 15 → 41 | 12 → 35 | 3 → 6 | 0 | +0 |
| Compact | 3 | 4 | 4 | 6 → 17 | 15 → 40 | 12 → 34 | 3 → 6 | 0 | +0 |
| One-hot | 0 | 8 | 8 | 7 → 55 | 34 → 126 | 24 → 82 | 10 → 44 | 8 | +24 |
| One-hot | 1 | 8 | 8 | 7 → 52 | 34 → 116 | 24 → 74 | 10 → 42 | 8 | +22 |
| One-hot | 3 | 8 | 8 | 7 → 55 | 34 → 119 | 24 → 77 | 10 → 42 | 8 | +22 |

| Encoding | Level | Initial placement | Final placement | Native gates used |
|---|---:|---|---|---|
| Compact | 0 | [0, 1, 2, 3] | [0, 1, 2, 3] | cx, rz, sx |
| Compact | 1 | [0, 1, 2, 3] | [0, 1, 2, 3] | cx, rz, sx |
| Compact | 3 | [0, 1, 2, 3] | [0, 1, 2, 3] | cx, rz, sx |
| One-hot | 0 | [0, 1, 2, 3, 4, 5, 6, 7] | [1, 0, 3, 2, 6, 4, 5, 7] | cx, rz, sx |
| One-hot | 1 | [0, 1, 2, 3, 4, 5, 6, 7] | [1, 0, 3, 2, 6, 4, 5, 7] | cx, rz, sx |
| One-hot | 3 | [0, 1, 2, 3, 4, 5, 6, 7] | [1, 0, 3, 2, 6, 4, 5, 7] | cx, rz, sx |

### Why these metrics matter on a QPU

**Why it matters here.** Available physical qubits limit placement. Connectivity determines where two-qubit gates can run and which states must move. Two-qubit gate fidelity describes how closely a gate matches its intended operation. More gates can expose the state to more error. Circuit depth counts sequential gate layers, while gate durations determine how much coherence time the circuit consumes. Measurement errors can also change the observed solution histogram, even when the circuit itself is correct. Increasing QAOA depth p expands the circuit family and generally increases exposure to noise.

## Takeaway

Logical and native two-qubit counts measure different things. Compare encodings under the same basis and topology, and use the all-connected native baseline to distinguish translation from connectivity overhead.

## Presentation asset

Project the logical RZZ beside its routed SWAP example, then show the before/after resource table with the routing columns.

MkDocs topic for this notebook: `docs/simulation.md`.

## Next step

Phase 10, `10_noisy_device_aware_aer.ipynb`, will compare ideal statevector predictions, finite-shot ideal Aer, and device-aware noisy Aer before the first QPU execution.